In [8]:
pip install decord transformers accelerate timm


Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
from pathlib import Path
import os
import torch
import torch.nn as nn
import timm
import random
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from decord import VideoReader, cpu
from tqdm import tqdm


VIDEO_ROOT = Path("/kaggle/input/datasets/indiff/videos")  
CSV_PATH = Path("/kaggle/input/datasets/indiff/labels/bah-video.csv")


df = pd.read_csv(CSV_PATH)


df["full_path"] = df["video-path"].apply(
    lambda x: str(VIDEO_ROOT / str(x).strip())
)

print("Проверка путей:\n")

for p in df["full_path"].sample(5):
    print(p)
    print("Exists:", os.path.exists(p))
    print("-" * 60)


Проверка путей:

/kaggle/input/datasets/indiff/videos/Videos/82662/Visite_1/82662_Question_4_2024-11-13_12-50-08_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82786/Visite_1/82786_Question_7_2024-12-12_13-24-21_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82690/Visite_1/82690_Question_6_2024-11-14_16-05-28_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/83008/Visite_1/83008_Question_4_2025-05-05_15-18-15_Video.mp4
Exists: True
------------------------------------------------------------
/kaggle/input/datasets/indiff/videos/Videos/82820/Visite_1/82820_Question_1_2025-02-02_04-58-42_Video.mp4
Exists: True
------------------------------------------------------------


In [6]:
print(df["video-path"].iloc[0])
print(df["full_path"].iloc[0])


Videos/82694/Visite_1/82694_Question_1_2024-11-15_21-05-54_Video.mp4
/kaggle/input/datasets/indiff/videos/Videos/82694/Visite_1/82694_Question_1_2024-11-15_21-05-54_Video.mp4


In [16]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
# Затем 30% делим пополам -> по 15% на Val и Test
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Размер выборок -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Размер выборок -> Train: 998 | Val: 214 | Test: 215


In [17]:
# Аугментации
train_transform = v2.Compose([
    v2.Resize((256, 256), antialias=True),
    v2.RandomResizedCrop(224, scale=(0.8, 1.0), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class RobustVideoDataset(Dataset):
    def __init__(self, dataframe, num_frames=16, transform=None, is_train=False):
        self.df = dataframe
        self.num_frames = num_frames
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def load_video(self, path):
        vr = VideoReader(path, ctx=cpu(0))
        total_frames = len(vr)
        
        if total_frames < self.num_frames:
            # Если видео слишком короткое, дублируем кадры
            indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        elif self.is_train:
            # Временной джиттеринг для аугментации
            max_offset = (total_frames - 1) // self.num_frames
            start = random.randint(0, max_offset) if max_offset > 0 else 0
            indices = np.linspace(start, total_frames - 1, self.num_frames).astype(int)
        else:
            indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)

        frames = vr.get_batch(indices).asnumpy()
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
        return frames

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video = self.load_video(row["full_path"])
        if self.transform:
            video = self.transform(video)
        label = torch.tensor(row["label"]).long()
        return video, label

train_loader = DataLoader(RobustVideoDataset(train_df, NUM_FRAMES, train_transform, is_train=True), 
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(RobustVideoDataset(val_df, NUM_FRAMES, val_test_transform, is_train=False), 
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(RobustVideoDataset(test_df, NUM_FRAMES, val_test_transform, is_train=False), 
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [12]:
class VideoTransformer(nn.Module):
    def __init__(self, num_classes=2, num_frames=16):
        super().__init__()
        # Тяжелый замороженный экстрактор
        self.backbone = timm.create_model('convnext_base', pretrained=True, num_classes=0)
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        feature_dim = self.backbone.num_features # 1024 для convnext_base
        embed_dim = 512
        
        self.proj = nn.Linear(feature_dim, embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_frames, embed_dim))
        self.dropout = nn.Dropout(0.3)
        
        # Временной Трансформер
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=1024, dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # Attention Pooling (ищет самые важные кадры)
        self.attention_pool = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # Классификатор
        self.fc = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.5),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.size()
        x_flat = x.view(B * T, C, H, W)
        
        with torch.no_grad():
            features = self.backbone(x_flat)
            
        features = features.view(B, T, -1)
        
        h = self.proj(features)
        h = h + self.pos_embedding[:, :T, :]
        h = self.dropout(h)
        
        h = self.transformer(h)
        
        attn_weights = torch.softmax(self.attention_pool(h), dim=1)
        pooled_features = torch.sum(h * attn_weights, dim=1)
        
        logits = self.fc(pooled_features)
        return logits

In [19]:
model = VideoTransformer(num_frames=NUM_FRAMES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_mf1 = 0.0
BATCH_SIZE = 4
EPOCHS = 15 
LR = 5e-4
NUM_FRAMES = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Фиксируем seed для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    train_loss = 0
    for videos, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        videos, labels = videos.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(videos)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
        optimizer.step()
        train_loss += loss.item()
        
    scheduler.step()
    avg_train_loss = train_loss / len(train_loader)
    
    # --- VALIDATION ---
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for videos, labels in val_loader:
            logits = model(videos.to(DEVICE))
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_true.extend(labels.numpy())
            
    val_acc = accuracy_score(val_true, val_preds)
    val_mf1 = f1_score(val_true, val_preds, average='macro')
    
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_mf1:.4f}")
    
    # Сохраняем лучшую модель по метрике на валидации
    if val_mf1 > best_val_mf1:
        best_val_mf1 = val_mf1
        torch.save(model.state_dict(), 'best_sota_model.pth')
        print(f" -> Модель улучшилась (Новый лучший Macro F1: {best_val_mf1:.4f}). Сохраняем веса!")

Epoch 1/15 [Train]: 100%|██████████| 250/250 [10:00<00:00,  2.40s/it]


Epoch 1 | Train Loss: 0.8451 | Val Acc: 0.4533 | Val Macro F1: 0.3119
 -> Модель улучшилась (Новый лучший Macro F1: 0.3119). Сохраняем веса!


Epoch 2/15 [Train]:  19%|█▉        | 48/250 [01:51<08:15,  2.45s/it]

In [15]:
print("\n=== ЗАПУСК ФИНАЛЬНОГО ТЕСТА НА ОТЛОЖЕННОЙ ВЫБОРКЕ (Ансамбль из 5 моделей) ===")
test_ds = RobustVideoDataset(test_df, num_frames=NUM_FRAMES, transform=val_test_transform, is_train=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

ensemble_preds = []

# Прогоняем тестовые данные через каждую из 5 сохраненных моделей
for fold in range(K_FOLDS):
    model.load_state_dict(torch.load(f'sota_model_fold_{fold+1}.pth'))
    model.eval()
    fold_probs = []
    
    with torch.no_grad():
        for videos, _ in tqdm(test_loader, desc=f"Predicting with Fold {fold+1}"):
            logits = model(videos.to(DEVICE))
            probs = torch.softmax(logits, dim=1) # Получаем вероятности
            fold_probs.append(probs.cpu().numpy())
            
    ensemble_preds.append(np.concatenate(fold_probs))

# Усредняем предсказания
avg_probs = np.mean(ensemble_preds, axis=0)
final_predictions = np.argmax(avg_probs, axis=1)
test_true_labels = test_df['label'].values

final_test_mf1 = f1_score(test_true_labels, final_predictions, average='macro')
final_test_acc = accuracy_score(test_true_labels, final_predictions)

print("\n" + "*"*50)
print(f"ИТОГОВАЯ ТОЧНОСТЬ (ACCURACY) НА ТЕСТЕ: {final_test_acc:.4f}")
print(f"ИТОГОВЫЙ MACRO F1 НА ТЕСТЕ: {final_test_mf1:.4f}")
print("*"*50)

NameError: name 'train_val_df' is not defined